# Result model

## Caricamento dataset

Come al solito, recuperiamo il dataset completo.

In [2]:
import pandas as pd
import numpy as np

path = "static/da-result/result.csv"
dataset = pd.read_csv(path)

## Merge delle feature

Raggruppiamo le coppie di feature Home/Away in un unica feature data dal differenza tra feature home e feature away. Questo oltre a ridurre il numero di colonne, ci aiuta anche a ridurre la ridondanza e a semplificare il calcolo dei differenziali ai modelli. 

In [3]:
# Creo feature uniche
dataset["WinStreak"] = dataset["Home_WinStreak"] - dataset["Away_WinStreak"]
dataset["Z_Goals_Season"] = dataset["Z_Home_Goals_Season"] - dataset["Z_Away_Goals_Season"]
dataset["Z_Wins_Season"] = dataset["Z_Home_Wins_Season"] - dataset["Z_Away_Wins_Season"]
dataset["GoalOnShotRatio"] = dataset["GoalOnShotRatioHome"] - dataset["GoalOnShotRatioAway"]
dataset["PointToMatchRatio"] = dataset["PointToMatchRatioHome"] - dataset["PointToMatchRatioAway"]
dataset["Elo"] = dataset["HomeElo"] - dataset["AwayElo"]

# Elimino le feature inutilizzate
dataset = dataset.drop(columns=["Home_WinStreak", "Away_WinStreak", "Z_Home_Goals_Season", "Z_Away_Goals_Season", "Z_Home_Wins_Season", "Z_Away_Wins_Season", "GoalOnShotRatioHome", "GoalOnShotRatioAway", "PointToMatchRatioHome", "PointToMatchRatioAway", "HomeElo", "AwayElo"])

dataset.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,FTR,Season,HomeAdvantage,WinStreak,Z_Goals_Season,Z_Wins_Season,GoalOnShotRatio,PointToMatchRatio,Elo
0,2015-08-22,Lazio,Bologna,2,1,H,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0
1,2015-08-22,Verona,Roma,1,1,D,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0
2,2015-08-23,Sassuolo,Napoli,2,1,H,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0
3,2015-08-23,Sampdoria,Carpi,5,2,H,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0
4,2015-08-23,Palermo,Genoa,1,0,H,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0


## Preparazione del target e split temporale

Procediamo:
- Mappando la colonna FTR in dati numerici (0 = Home, 1 = Draw, 2=Away)
- Ordinando cronologicamente il dataset
- Partizionandolo cronologicamente in set di training e set di test

In [4]:
from sklearn.metrics import accuracy_score, classification_report

# Mappo il target FTR in numeri
map = {'H': 0, 'D': 1, 'A': 2}
dataset['Numeric_target'] = dataset['FTR'].map(map)

# Ordino il dataset
dataset = dataset.sort_values(by='Date').reset_index(drop=True)

# Divido il traning set e test set in base alle stagioni (temporale)
train_mask = dataset['Season'] < '2024-2025'
test_mask = dataset['Season'] >= '2024-2025'

# Isolo il target per training e per test
Y_train = dataset.loc[train_mask, 'Numeric_target']
Y_test = dataset.loc[test_mask, 'Numeric_target']

X = dataset.drop(columns=['FTR'])
X_train = X.loc[train_mask]
X_test = X.loc[test_mask]
X_train.head()

,Date,HomeTeam,AwayTeam,FTHG,FTAG,Season,HomeAdvantage,WinStreak,Z_Goals_Season,Z_Wins_Season,GoalOnShotRatio,PointToMatchRatio,Elo,Numeric_target
0,2015-08-22,Lazio,Bologna,2,1,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,0
1,2015-08-22,Verona,Roma,1,1,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,1
2,2015-08-23,Sassuolo,Napoli,2,1,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,0
3,2015-08-23,Sampdoria,Carpi,5,2,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,0
4,2015-08-23,Palermo,Genoa,1,0,2015-2016,0.0,0,0.0,0.0,0.0,0.0,0.0,0


### Random Baseline

Cominciamo osservando come un esperimento casuale si comporterebbe davanti alle partite del nostro test set (760). Questo passaggio lo utilizzamo per simulare lo scenario in cui un decisore (computer o un dado a tre facce), totalmente privo di dati storici o competenze calcistiche, tenti di indovinare l'esito dei match affidandosi al puro caso. 
L'output di questa simulazione fissa la nostra linea di partenza (random baseline) e diventa fondamentale per valutare il reale valore della ricerca.  

In [5]:
# Imposto il seed 
np.random.seed(42)

# Genero le predizioni totalmente casuali per le partite del test set
preds_casuali = np.random.choice([0, 1, 2], size=len(Y_test))

# Calcolo l'accuratezza del puro caso
accuratezza_puro_caso = accuracy_score(Y_test, preds_casuali)


print("\n==================================")
print("METRICHE DELL'ESPERIMENTO CASUALE")
print("==================================")
print(f"Accuratezza puro caso:    {accuratezza_puro_caso:.4f} ({accuratezza_puro_caso*100:.2f}%)")
print("\nReport dettagliato del Modello Casuale Puro:")
print(classification_report(Y_test, preds_casuali, target_names=['1', 'X', '2'], zero_division=0))


METRICHE DELL'ESPERIMENTO CASUALE
Accuratezza puro caso:    0.3132 (31.32%)

Report dettagliato del Modello Casuale Puro:
              precision    recall  f1-score   support

           1       0.39      0.35      0.37       299
           X       0.26      0.30      0.28       207
           2       0.28      0.27      0.28       254

    accuracy                           0.31       760
   macro avg       0.31      0.31      0.31       760
weighted avg       0.32      0.31      0.31       760



## Scelta delle feature
In questa sezione viene creata una classe che, dati in input le feature di interesse, utilizza l'interpolazione logistica per calcolare le metriche del modello, in funzione delle feautre che utilizziamo ingresso.

In [ ]:
from pandas import DataFrame    
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, log_loss, classification_report

class LinearRegressionTester:
    
    __train_set: DataFrame
    __test_set: DataFrame

    def __init__(self, train_set: DataFrame, test_set: DataFrame):
        
        self.__train_set = train_set
        self.__test_set = test_set
    
   
    def run_logistic_regression_baseline(self, target_col: str, baseline_features: list[str]):
        """
        Esegue la baseline lineare (Regressione Logistica) utilizzando le feature scelte.
        
        :param target_col: Il nome della colonna contenente le etichette target (Y)
        :param baseline_features: lista di feature con le quali viene valutato il modello
        """

        # Estrazione delle feature di nostro interesse
        X_train_features = self.__train_set[baseline_features]
        Y_train = self.__train_set[target_col]
        
        X_test_features = self.__test_set[baseline_features]
        Y_test = self.__test_set[target_col]

        # Standardizzaziones
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train_features)
        X_test_scaled = scaler.transform(X_test_features)

        # Addestramento del Modello
        log_model = LogisticRegression(
            solver='lbfgs', 
            max_iter=500, 
            random_state=42,
            class_weight="balanced"
        )
        log_model.fit(X_train_scaled, Y_train)

        # Predizioni delle probabilità sul Test Set
        base_preds = log_model.predict(X_test_scaled)
        base_probs = log_model.predict_proba(X_test_scaled)

        print(f"Accuratezza (Precision): {accuracy_score(Y_test, base_preds):.4f} ({accuracy_score(Y_test, base_preds)*100:.2f}%)")
        print(f"Log-Loss:    {log_loss(Y_test, base_probs):.4f}")
        print("\nReport:")
        print(classification_report(Y_test, base_preds, target_names=['1', 'X', '2'], zero_division=0))
        

## Scelta delle feature con regressione logistica
Adesso, tramite la classe appena creata, scegliamo delle feature e valutiamo l'impatto sulle metriche, eliminandone e aggiungendone di nuove, seguendo anche la heatmap del notebook feature-choice. Per prima cosa creaiamo un'istanza dalla classe, che successivamente andremo ad utilizzare.

In [7]:
ml = LinearRegressionTester(X_train, X_test)
baseline = ["WinStreak","PointToMatchRatio","GoalOnShotRatio","HomeAdvantage"] # Caratteristiche base

### Baseline con regressione logistica 

Realizziamo adesso una baseline basata sul modello della regressione logistica. Per farlo utilizziamo delle feature base da noi calcolate, come la striscia di vittorie, il rapporto punti/match ed altri. Notiamo ovviamente un netto miglioramento rispetto alla random baseline, ma anche un limite importante riguardo i pareggi. 
L'algoritmo infatti impara cercando di minimizzare la loss-function. Nel nostro dataset essendo che i pareggi sono la minoranza (come sempre nel calcio), la regressione logistica nota che se prova a predire un pareggio ma la partita finisce diversamente subisce una penalizzazione importante sulla Log-Loss. Quindi per fare meno errori possibili evita di predirli. 

In [8]:
ml.run_logistic_regression_baseline("Numeric_target", baseline)

Accuratezza (Precision): 0.5066 (50.66%)
Log-Loss:    1.0188

Report:
              precision    recall  f1-score   support

           1       0.48      0.78      0.60       299
           X       0.00      0.00      0.00       207
           2       0.55      0.59      0.57       254

    accuracy                           0.51       760
   macro avg       0.34      0.46      0.39       760
weighted avg       0.37      0.51      0.43       760



### Aggiunta dello z-score
Aggiungiamo alla baseline, le caratteristiche dello z-score, ovvero le feature che indicano lo scostamento in deviazioni standard, dalla media della specifica caratteristica nel campionato. Per prima cosa inseriamo entrambe le feature Z_Wins_Season, per vedere cosa succede

In [9]:
ml.run_logistic_regression_baseline("Numeric_target", baseline + ["Z_Wins_Season", "Z_Goals_Season"])

Accuratezza (Precision): 0.5092 (50.92%)
Log-Loss:    1.0147

Report:
              precision    recall  f1-score   support

           1       0.48      0.80      0.60       299
           X       0.00      0.00      0.00       207
           2       0.56      0.59      0.58       254

    accuracy                           0.51       760
   macro avg       0.35      0.46      0.39       760
weighted avg       0.38      0.51      0.43       760



notiamo in questo caso che aggiungendo una feature l'accuratezza generale decresce, mentre la log loss sale. Questo fenomeno ha una spiegazione che possiamo dimostrare con la heatmap generata in precedenza: le feature Z-Score sono **ridondanti** rispetto ad altre feature, ovvero possiedono una correlazione molto alta (la Z_Wins_Season ha una correlazione del 95% con PointToMatchRatio). Questo dettaglio porta il modello a peggiorare le sue performance. Eliminoamo Z_Wins_Season, lasciando solamente Z_Goals_Season, che notiamo avere un'elevata correlazione solo con PointToMatchRatio (ma molto inferiore al 95% precedente). 

In [10]:
ml.run_logistic_regression_baseline("Numeric_target", baseline + ["Z_Goals_Season"])

Accuratezza (Precision): 0.5092 (50.92%)
Log-Loss:    1.0145

Report:
              precision    recall  f1-score   support

           1       0.48      0.80      0.60       299
           X       0.00      0.00      0.00       207
           2       0.56      0.59      0.58       254

    accuracy                           0.51       760
   macro avg       0.35      0.46      0.39       760
weighted avg       0.38      0.51      0.43       760



notiamo in questo caso che la precisione è rimansta invariata rispetto al primo caso. Questo è indice del fatto che l'aggiunta di una nuova feature (poco ridondante rispetto alle altre), non porta ad uno svantaggio statistico.